# LUMI 정지 이미지 규칙 패턴 스크리닝 (`static_pattern_screening.ipynb`)

`image_discriminator_1.ipynb`(v1)의 리뷰 결과를 반영한 개선판이다. **이 노트북 하나에 계산 코드가 모두 들어 있어** 저장소 없이 단독으로 실행된다. 필요 패키지: `numpy`, `opencv-python`, `pandas`.

| 항목 | v1 | v2 |
|---|---|---|
| 줄무늬 쌍 계산 | `경계 수 // 2` → 항상 1쌍 누락 (5쌍 → 4쌍) | 줄무늬 수를 직접 셈 (정확히 5쌍 → 5쌍) |
| 패턴 면적 | 쌍 개수(중앙 선)와 면적(화면 전체 대비)을 따로 계산 | **5쌍 이상이 검출된 구간만** 면적에 포함, 인접 주사선 일관성 확인 |
| 탐색 범위 | 화면 중앙을 지나는 선만 | 화면 전체 가로·세로 주사선 (세로·가로·사선·동심원·방사형) |
| 고대비 / 고휘도 / 적색 | 정지 화면에 깜빡임 기준을 적용해 '위험' 판정 | 판정하지 않음 → 시간축 판정에 쓸 **특징값** |
| 확률 3종 | '확률', '환자 위험 확률' | '확률' 제거. Binnie 곡선은 시청 기하를 입력받는 **참고값** |
| 해상도 | 1080px로 축소 (가는 줄무늬 왜곡) | 원본 해상도 (4K 초과만 축소하고 기록) |
| 입력 | 고정 경로 `C:\LUMI\이미지\패턴.png` | `IMAGE_PATH` 지정, 없으면 합성 예제 세트 |

> **안전:** 이 노트북은 입력 이미지를 화면에 표시하지 않는다. 결과는 수치 표와 패턴 영역 마스크(흑백 면적) 파일뿐이다. 출력은 연구용 스크리닝 값이며 ITU-R BT.1702 / WCAG 적합성 판정이나 임상적 안전성 판단이 아니다.

## 1. 설정과 상수

휘도 가정(흰색 200 cd/m²), ITU-R BT.1702 명암 대비 분기점(160 cd/m²), 패턴 기준(명암 5쌍)과 검출 매개변수.

In [1]:
from __future__ import annotations

import math
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np

PEAK_WHITE_CD = 200.0          # SDR 흰색 가정 (cd/m²)
DARK_LIMIT_CD = 160.0          # ITU-R BT.1702 명암 대비 규칙의 분기점
MIN_PAIRS = 5                  # 규칙적 패턴: 명암 쌍 5개 이상
SAME_POLARITY_RATIO = 1.75     # 같은 극성(밝은-밝은, 어두운-어두운) 줄무늬 폭 허용 비
LINE_COHERENCE = 0.6           # 인접 주사선 5개 중 이 비율 이상에서 함께 검출돼야 인정
MAX_SIDE = 3840

## 2. 입력과 휘도 변환

한글 경로를 읽을 수 있는 이미지 로더, sRGB → 선형 RGB(IEC 61966-2-1), 휘도 `L = 200 × (0.2126R + 0.7152G + 0.0722B)` cd/m², 명암 대비 규칙(어두운 쪽 <160 cd/m²이면 ΔL ≥ 20, 아니면 Michelson > 1/17).

In [2]:
def load_image_rgb(path: str | Path) -> np.ndarray:
    """한글 경로를 지원하는 이미지 로더. (H, W, 3) uint8 RGB."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"입력 파일을 찾을 수 없습니다: {path}")
    bgr = cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f"이미지를 읽을 수 없습니다(정지 이미지 전용): {path}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def srgb_to_linear(rgb_u8: np.ndarray) -> np.ndarray:
    """IEC 61966-2-1 sRGB 감마 복원. 결과 [0, 1] float32."""
    c = rgb_u8.astype(np.float32) / 255.0
    return np.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4).astype(np.float32)


def luminance_cd(linear_rgb: np.ndarray, peak_white: float = PEAK_WHITE_CD) -> np.ndarray:
    """선형 RGB → 휘도(cd/m²). 표시 장치 최대 밝기를 `peak_white`로 가정한다."""
    y = 0.2126 * linear_rgb[..., 0] + 0.7152 * linear_rgb[..., 1] + 0.0722 * linear_rgb[..., 2]
    return (peak_white * y).astype(np.float32)


def is_contrast_pair(high: float, low: float) -> bool:
    """ITU-R BT.1702 명암 대비 규칙: 어두운 쪽 <160이면 ΔL≥20, 아니면 Michelson>1/17."""
    if low < DARK_LIMIT_CD:
        return high - low >= 20.0
    return (high - low) / (high + low) > 1.0 / 17.0

## 3. 1차원 줄무늬 계수

한 줄(주사선)의 밝기 변화에서 **규칙적인 명암 줄무늬 구간**을 찾는다.

1. 명암 대비 규칙을 만족할 때만 방향이 바뀌는 교대 극값(밝음·어두움 꼭짓점)을 찾는다.
2. 이웃 극값 사이의 중간 밝기 지점을 경계로 삼아 줄무늬 폭을 구한다.
3. 같은 극성 줄무늬(밝은-밝은, 어두운-어두운)의 폭 비가 1.75배 이내면 같은 패턴으로 묶는다. 흰 바탕 위 가는 검은 선처럼 명/암 폭이 달라도 허용된다.
4. **쌍 = 줄무늬 수 // 2.** v1은 `경계 수 // 2`로 계산해 항상 1쌍이 모자랐다.
5. 보수적 계수: 패턴을 둘러싼 넓은 바탕도 경계 줄무늬 1개로 센다(회색 바탕 4쌍 → 5쌍 가능).

In [3]:
def _alternating_extrema(p: np.ndarray) -> list[int]:
    """인접한 두 극값이 항상 명암 대비 규칙을 만족하는 교대 극값의 인덱스."""
    n = len(p)
    if n < 2:
        return []
    out: list[int] = []
    hi = lo = 0
    direction = 0      # 0: 미정, 1: 최댓값 추적 중, -1: 최솟값 추적 중
    cand = 0
    for i in range(1, n):
        v = p[i]
        if direction == 0:
            if v > p[hi]:
                hi = i
            if v < p[lo]:
                lo = i
            if is_contrast_pair(p[hi], p[lo]):
                if hi < lo:
                    out.append(hi); direction = -1; cand = lo
                else:
                    out.append(lo); direction = 1; cand = hi
        elif direction == 1:
            if v > p[cand]:
                cand = i
            elif is_contrast_pair(p[cand], v):
                out.append(cand); direction = -1; cand = i
        else:
            if v < p[cand]:
                cand = i
            elif is_contrast_pair(v, p[cand]):
                out.append(cand); direction = 1; cand = i
    if direction != 0:
        out.append(cand)
    return out


def _edge_between(p: np.ndarray, a: int, b: int) -> int:
    """극값 a→b 사이에서 중간 밝기를 처음 넘는 위치."""
    mid = (p[a] + p[b]) / 2.0
    seg = p[a:b + 1]
    hits = np.flatnonzero(seg <= mid) if p[a] > p[b] else np.flatnonzero(seg >= mid)
    return a + int(hits[0]) if hits.size else b


def _plateau_start(p: np.ndarray, i0: int, mid: float, high: bool) -> int:
    side = p[:i0 + 1] >= mid if high else p[:i0 + 1] <= mid
    off = np.flatnonzero(~side)
    return int(off[-1]) + 1 if off.size else 0


def _plateau_end(p: np.ndarray, i1: int, mid: float, high: bool) -> int:
    side = p[i1:] >= mid if high else p[i1:] <= mid
    off = np.flatnonzero(~side)
    return i1 + (int(off[0]) if off.size else len(side))


@dataclass(frozen=True)
class StripeRun:
    start: int      # 첫 줄무늬 시작 픽셀
    end: int        # 마지막 줄무늬 끝 픽셀(미포함)
    stripes: int    # 밝은/어두운 줄무늬 개수

    @property
    def pairs(self) -> int:
        return self.stripes // 2


def stripe_runs(profile: np.ndarray) -> list[StripeRun]:
    """1D 휘도 프로파일에서 폭이 규칙적인 명암 줄무늬 구간을 찾는다.

    - 줄무늬 = 대비 규칙을 만족하는 교대 극값 하나를 중심으로 한 띠
    - 같은 극성 줄무늬(k와 k-2)의 폭 비가 `SAME_POLARITY_RATIO` 이하이면 같은 구간
      (흰 바탕 위 가는 검은 선처럼 명/암 폭이 달라도 허용)
    - 영상 경계에서 잘린 끝 줄무늬는 더 좁은 경우만 허용, 넓은 바탕은 줄무늬로 세지 않음
    """
    p = np.asarray(profile, dtype=np.float32)
    ext = _alternating_extrema(p)
    if len(ext) < 2:
        return []
    edges = [_edge_between(p, ext[k], ext[k + 1]) for k in range(len(ext) - 1)]
    n = len(p)
    first_high = p[ext[0]] > p[ext[1]]
    mid0 = (p[ext[0]] + p[ext[1]]) / 2.0
    midn = (p[ext[-2]] + p[ext[-1]]) / 2.0
    last_high = p[ext[-1]] > p[ext[-2]]
    lefts = [_plateau_start(p, ext[0], mid0, first_high)] + edges
    rights = edges + [_plateau_end(p, ext[-1], midn, last_high)]
    widths = [max(1, r - l) for l, r in zip(lefts, rights)]
    truncated = {0: lefts[0] == 0, len(widths) - 1: rights[-1] >= n}

    def consistent(k: int) -> bool:
        a, b = widths[k], widths[k - 2]
        if truncated.get(k) and a <= b:
            return True
        if truncated.get(k - 2) and b <= a:
            return True
        return max(a, b) / min(a, b) <= SAME_POLARITY_RATIO

    runs: list[StripeRun] = []
    start = 0
    for k in range(2, len(widths) + 1):
        if k == len(widths) or not consistent(k):
            # k-2와 k가 어긋나면 구간 [start, k-1]을 닫고 k-1부터 새 구간
            end_k = k - 1
            if end_k - start + 1 >= 2:
                # 보수적 계수: 구간을 둘러싼 넓은 바탕(폭 불일치로 제외된 줄무늬)도
                # 경계 줄무늬 1개로 센다. 흰 바탕 위 패턴의 끝 흰 줄무늬가 바탕과
                # 합쳐져 1쌍 적게 세는 것을 막는다(과소 계수보다 과대 계수를 택함).
                # 부작용: 줄무늬와 다른 제3의 밝기 바탕(예: 회색)은 실제 경계 줄무늬로
                # 볼 수 있어 1쌍 많게 셀 수 있다(회색 바탕 4쌍 → 5쌍).
                bounds = int(start > 0) + int(end_k < len(widths) - 1)
                runs.append(StripeRun(lefts[start], rights[end_k], end_k - start + 1 + bounds))
            start = k - 1
    return runs


def count_stripe_pairs(profile: np.ndarray) -> int:
    """프로파일 안의 가장 긴 규칙 구간의 명암 쌍 수."""
    return max((r.pairs for r in stripe_runs(profile)), default=0)

## 4. 2차원 패턴 영역

- 가로 주사선은 세로·사선·동심원·방사형 줄무늬를, 세로 주사선은 가로 줄무늬를 잡는다.
- 한 줄에서 **5쌍 이상인 구간만** 표시하고, 인접 주사선 5줄 중 60% 이상에서 함께 검출된 위치만 남긴다(글자·잡음 제거).
- **줄무늬 쌍 수와 패턴 면적을 같은 영역에서 계산**한다. v1은 쌍 수(화면 중앙 선)와 면적(화면 전체 대비)을 따로 계산해, 작은 줄무늬 조각 + 주변 글자가 '40% 이상 패턴'으로 판정됐다.

In [4]:
def _scan_mask(lum: np.ndarray, step: int, min_pairs: int):
    """행 방향 주사선마다 min_pairs 이상인 줄무늬 구간을 표시.

    반환: (표본행 × W) 마스크, 표시한 구간 목록 [(표본행 i, StripeRun)], 단일 주사선 최대 쌍 수
    """
    rows = range(0, lum.shape[0], step)
    marks = np.zeros((len(rows), lum.shape[1]), dtype=bool)
    marked: list[tuple[int, StripeRun]] = []
    single_best = 0
    for i, y in enumerate(rows):
        for run in stripe_runs(lum[y]):
            single_best = max(single_best, run.pairs)
            if run.pairs >= min_pairs:
                marks[i, run.start:run.end] = True
                marked.append((i, run))
    return marks, marked, single_best


def _coherent_best(coherent: np.ndarray, marked: list[tuple[int, StripeRun]]) -> int:
    """일관성 필터를 통과한 영역(구간의 절반 이상)에 속한 구간 중 최대 쌍 수."""
    return max((run.pairs for i, run in marked
                if coherent[i, run.start:run.end].mean() >= 0.5), default=0)


def _coherent(marks: np.ndarray, window: int = 5) -> np.ndarray:
    """인접 주사선에서도 함께 검출된 위치만 남긴다(글자·잡음의 우연한 일치 제거)."""
    if marks.shape[0] < window:
        return marks
    pad = window // 2
    padded = np.pad(marks.astype(np.float32), ((pad, pad), (0, 0)))
    c = np.cumsum(padded, axis=0)
    c = np.vstack([np.zeros((1, c.shape[1]), np.float32), c])
    mean = (c[window:] - c[:-window]) / window
    return marks & (mean >= LINE_COHERENCE)


def _expand(marks: np.ndarray, step: int, length: int) -> np.ndarray:
    return np.repeat(marks, step, axis=0)[:length]


@dataclass
class PatternResult:
    max_pairs: int              # 패턴 영역(일관성 통과)에 속한 줄무늬 구간의 최대 쌍 수
    max_pairs_single_line: int  # 진단용: 필터 전 단일 주사선 최대값 (글자·잡음 포함)
    area_fraction: float
    mask: np.ndarray            # analyzed_shape 크기 bool — 5쌍 이상 줄무늬가 검출된 픽셀
    analyzed_shape: tuple[int, int]


def detect_regular_pattern(lum: np.ndarray, *, min_pairs: int = MIN_PAIRS,
                           max_lines: int = 270) -> PatternResult:
    """가로·세로 주사선으로 규칙적 명암 줄무늬 영역을 찾는다.

    가로 주사선은 세로·사선·동심원·방사형 줄무늬를, 세로 주사선은 가로 줄무늬를 잡는다.
    주사선 간격(step)이 커지는 큰 영상에서는 높이(폭)가 약 3×step 미만인 패턴이
    일관성 필터에서 제외될 수 있다.
    """
    h, w = lum.shape
    step_r = max(1, math.ceil(h / max_lines))
    step_c = max(1, math.ceil(w / max_lines))
    row_marks, row_runs, single_r = _scan_mask(lum, step_r, min_pairs)
    col_marks, col_runs, single_c = _scan_mask(lum.T, step_c, min_pairs)
    row_ok, col_ok = _coherent(row_marks), _coherent(col_marks)
    mask = _expand(row_ok, step_r, h) | _expand(col_ok, step_c, w).T
    best = max(_coherent_best(row_ok, row_runs), _coherent_best(col_ok, col_runs))
    return PatternResult(best, max(single_r, single_c), float(mask.mean()), mask, (h, w))

## 5. 특징값 (판정 아님)

고휘도·포화 적색은 표준에서 프레임 사이 *변화*(깜빡임)의 조건이다. 정지 이미지 한 장으로는 위험을 판정하지 않고, 다음 단계 시간축 판정기의 입력으로 쓸 수치만 낸다.

In [5]:
def saturated_red_fraction(linear_rgb: np.ndarray) -> float:
    """WCAG 포화 적색 정의 R/(R+G+B) ≥ 0.8 (선형 RGB). 판정이 아닌 면적 특징."""
    s = linear_rgb.sum(axis=-1)
    red = linear_rgb[..., 0]
    ratio = np.divide(red, s, out=np.zeros_like(red), where=s > 1e-4)
    return float(((ratio >= 0.8) & (red > 0.01)).mean())


def luminance_features(lum: np.ndarray) -> dict[str, float]:
    p5, p50, p95 = np.percentile(lum, [5, 50, 95])
    return {"mean_cd": float(lum.mean()), "p05_cd": float(p5), "p50_cd": float(p50),
            "p95_cd": float(p95), "bright_area_ge160": float((lum >= DARK_LIMIT_CD).mean())}

## 6. 시청 기하 참고값 (확률 아님)

Binnie 곡선 `Q = 100(1 − e^(−0.0574θ))`, 비율 `0.021Q − 0.1827`(Q ∈ [8.75, 56.37]). θ는 화면 크기·시청 거리로 계산한다. **θ를 '패턴과 같은 넓이의 원 지름 시각도'로 본 것은 가정**이며 Jordan & Vanderheiden (2024) 부록 A와 대조가 필요하다.

In [6]:
@dataclass(frozen=True)
class ViewingGeometry:
    diagonal_inch: float = 27.0
    aspect: float = 16 / 9
    distance_cm: float = 60.0

    def screen_cm(self) -> tuple[float, float]:
        d = self.diagonal_inch * 2.54
        h = d / math.sqrt(1 + self.aspect ** 2)
        return h * self.aspect, h


def pattern_visual_angle_deg(area_fraction: float, geometry: ViewingGeometry) -> float:
    """패턴 면적을 같은 넓이의 원으로 보고 그 지름이 이루는 시각도.

    가정: Binnie 관계식의 θ를 '자극의 등가 원 지름 시각도'로 해석했다.
    Jordan & Vanderheiden(2024) 부록 A 원문과 대조가 필요하다.
    """
    if area_fraction <= 0:
        return 0.0
    w, h = geometry.screen_cm()
    diameter = 2.0 * math.sqrt(area_fraction * w * h / math.pi)
    return math.degrees(2.0 * math.atan(diameter / (2.0 * geometry.distance_cm)))


def binnie_reference(theta_deg: float) -> dict[str, float]:
    """Q = 100(1 - e^(-0.0574θ)), 비율 = 0.021Q - 0.1827 (Q∈[8.75, 56.37]에서 0→1).

    광과민 환자 중 '패턴 자극'에 뇌파 이상 반응을 보이는 비율의 참고 곡선이다.
    발작 확률이 아니며 고휘도·적색 자극에는 적용하지 않는다.
    """
    q = 100.0 * (1.0 - math.exp(-0.0574 * theta_deg))
    prop = 0.0 if q <= 8.75 else 1.0 if q >= 56.37 else 0.021 * q - 0.1827
    return {"theta_deg": theta_deg, "Q": q, "binnie_reference": float(min(max(prop, 0.0), 1.0))}

## 7. 통합 분석 함수

In [7]:
def analyze_image(rgb_u8: np.ndarray, *, area_thresholds: tuple[float, ...] = (0.25, 0.40),
                  geometry: ViewingGeometry | None = None,
                  peak_white: float = PEAK_WHITE_CD) -> dict:
    """정지 이미지 1장의 패턴 스크리닝 + 공간 특징. 결과에 영상(픽셀)은 포함하지 않는다(mask 제외)."""
    geometry = geometry or ViewingGeometry()
    resized = False
    h, w = rgb_u8.shape[:2]
    if max(h, w) > MAX_SIDE:
        scale = MAX_SIDE / max(h, w)
        size = (max(1, round(w * scale)), max(1, round(h * scale)))
        rgb_u8 = cv2.resize(rgb_u8, size, interpolation=cv2.INTER_AREA)
        resized = True
    lin = srgb_to_linear(rgb_u8)
    lum = luminance_cd(lin, peak_white)
    pattern = detect_regular_pattern(lum)
    mask = pattern.mask
    if resized:  # 마스크를 입력 크기로 되돌려 원본과 겹쳐 볼 수 있게 한다
        mask = cv2.resize(mask.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST).astype(bool)
    screening = {f"pattern_over_{int(round(t * 100))}pct": bool(
        pattern.max_pairs >= MIN_PAIRS and pattern.area_fraction > t) for t in area_thresholds}
    return {
        "analysis": "lumi.static_features v2 (research screening, not conformance)",
        "input_shape": (h, w), "analyzed_shape": pattern.analyzed_shape, "resized": resized,
        "pattern": {"max_pairs": pattern.max_pairs,
                    "max_pairs_single_line": pattern.max_pairs_single_line,
                    "area_fraction": pattern.area_fraction,
                    **screening},
        "features": {**luminance_features(lum),
                     "saturated_red_area": saturated_red_fraction(lin)},
        "reference": binnie_reference(pattern_visual_angle_deg(pattern.area_fraction, geometry)),
        "geometry": {"diagonal_inch": geometry.diagonal_inch, "aspect": geometry.aspect,
                     "distance_cm": geometry.distance_cm, "peak_white_cd": peak_white},
        "mask": mask,               # 입력 이미지와 같은 (H, W)
    }

## 8. 실행

`IMAGE_PATH`에 이미지 경로를 넣으면 그 이미지를, `None`이면 v1 리뷰에서 오판을 재현한 사례가 포함된 합성 예제 7종을 분석한다.

In [8]:
IMAGE_PATH = None   # 예: r"C:\LUMI\이미지\패턴.png"
GEOMETRY = ViewingGeometry(diagonal_inch=27, distance_cm=60)   # 27인치 모니터, 60cm
MASK_DIR = Path("static_masks")   # 패턴 영역 마스크 저장 폴더 (노트북 기준 상대 경로)

In [9]:
def demo_images(h: int = 540, w: int = 960) -> dict[str, np.ndarray]:
    """실제 패턴을 적용한 알고리즘."""
    yy, xx = np.mgrid[:h, :w]
    gray = lambda g: np.repeat((g * 255).astype(np.uint8)[..., None], 3, axis=2)
    out = {}

    doc = np.full((h, w, 3), 255, np.uint8)
    words = "The quick brown fox jumps over the lazy dog LUMI research 2026".split()
    rng = np.random.default_rng(1)
    for y in range(36, h - 10, 26):
        x = 30
        while True:
            word = words[rng.integers(len(words))]
            (tw, _), _ = cv2.getTextSize(word, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
            if x + tw > w - 30:
                break
            cv2.putText(doc, word, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (20, 20, 20), 1, cv2.LINE_AA)
            x += tw + 10
    out["흰 배경 문서"] = doc

    noisy = np.full((h, w, 3), 255, np.uint8)
    for _ in range(1500):
        y, x = rng.integers(0, h - 6), rng.integers(0, w - 12)
        noisy[y:y + 5, x:x + rng.integers(3, 12)] = 0
    cy, cx = h // 2, w // 2
    noisy[cy - 45:cy + 45, cx - 90:cx + 90] = 255
    for i in range(0, 180, 12):
        noisy[cy - 45:cy + 45, cx - 90 + i:cx - 84 + i] = 0
    out["작은 줄무늬 조각 + 글자형 블록"] = noisy

    out["좌흑 / 우백"] = gray(xx >= w // 2)
    red = np.full((h, w, 3), 255, np.uint8); red[:, : int(w * 0.45)] = (255, 0, 0)
    out["포화 적색 45% (정지)"] = red
    out["세로 줄무늬 전체"] = gray((xx // 6) % 2)
    out["동심원 전체"] = gray((np.hypot(yy - h / 2, xx - w / 2) // 10) % 2)
    half = np.full((h, w, 3), 128, np.uint8)
    for x in range(0, w // 2, 10):
        half[:, x:x + 5] = 255; half[:, x + 5:x + 10] = 0
    out["화면 왼쪽 절반 줄무늬"] = half
    return out

images = {Path(IMAGE_PATH).name: load_image_rgb(IMAGE_PATH)} if IMAGE_PATH else demo_images()
print(f"분석 대상 {len(images)}개:", ", ".join(images))

분석 대상 7개: 흰 배경 문서, 작은 줄무늬 조각 + 글자형 블록, 좌흑 / 우백, 포화 적색 45% (정지), 세로 줄무늬 전체, 동심원 전체, 화면 왼쪽 절반 줄무늬


In [10]:
import pandas as pd

results, rows = {}, []
for name, img in images.items():
    r = analyze_image(img, geometry=GEOMETRY)
    results[name] = r
    p, f, ref = r["pattern"], r["features"], r["reference"]
    rows.append({
        "이미지": name,
        "최대 줄무늬 쌍": p["max_pairs"],
        "패턴 면적 %": round(p["area_fraction"] * 100, 1),
        "패턴 >25%": "위험" if p["pattern_over_25pct"] else "-",
        "패턴 >40%": "위험" if p["pattern_over_40pct"] else "-",
        "평균 휘도 cd/m²": round(f["mean_cd"], 1),
        "160cd/m² 이상 면적 %": round(f["bright_area_ge160"] * 100, 1),
        "포화 적색 면적 %": round(f["saturated_red_area"] * 100, 1),
        "패턴 시각도 °": round(ref["theta_deg"], 1),
        "Binnie 참고값": round(ref["binnie_reference"], 2),
    })
table = pd.DataFrame(rows).set_index("이미지")
table

,최대 줄무늬 쌍,패턴 면적 %,패턴 >25%,패턴 >40%,평균 휘도 cd/m²,160cd/m² 이상 면적 %,포화 적색 면적 %,패턴 시각도 °,Binnie 참고값
이미지,,,,,,,,,
흰 배경 문서,6,0.8,-,-,178.5,85.8,0.0,4.2,0.27
작은 줄무늬 조각 + 글자형 블록,17,3.5,-,-,178.1,89.1,0.0,9.0,0.66
좌흑 / 우백,0,0.0,-,-,100.0,50.0,0.0,0.0,0.00
포화 적색 45% (정지),0,0.0,-,-,129.1,55.0,45.0,0.0,0.00
세로 줄무늬 전체,80,100.0,위험,위험,100.0,50.0,0.0,45.7,1.00
동심원 전체,48,99.9,위험,위험,100.0,50.0,0.0,45.7,1.00
화면 왼쪽 절반 줄무늬,48,50.0,위험,위험,71.6,25.0,0.0,33.2,1.00


## 결과 읽는 법

- **판정은 '패턴 >25%', '패턴 >40%' 두 열뿐이다.** 조건: 같은 영역에서 명암 대비 규칙(ΔL≥20 cd/m², 어두운 쪽 ≥160이면 Michelson>1/17)을 만족하는 줄무늬가 **5쌍 이상**이고, 그런 영역의 면적이 기준을 넘을 때.
- **휘도·적색 열은 특징값**이다. 표준의 20 cd/m²·적색비 0.8·Δu′v′ 0.2는 프레임 사이 *변화*(깜빡임)의 조건이므로, 정지 이미지 한 장으로는 위험을 판정하지 않는다. 다음 단계의 시간축 판정기 입력으로 쓴다.
- **Binnie 참고값**은 패턴 자극에 대한 광과민 환자의 뇌파 반응 비율 곡선을 시청 기하에 대입한 값이다. 발작 확률이 아니다. θ를 '패턴 면적과 같은 넓이의 원 지름 시각도'로 해석한 것은 가정이며, Jordan & Vanderheiden (2024) 부록 A 원문과 대조가 필요하다.
- **보수적 계수:** 패턴을 둘러싼 넓은 바탕도 경계 줄무늬 1개로 센다. 그래서 회색 바탕 위 4쌍은 5쌍으로 셀 수 있다(과소 계수보다 과대 계수를 택함).

In [11]:
# 패턴 영역 마스크만 저장 (입력 이미지는 저장·표시하지 않음)
mask_dir = MASK_DIR
mask_dir.mkdir(parents=True, exist_ok=True)
for i, (name, r) in enumerate(results.items()):
    path = mask_dir / f"{i:02d}_mask.png"
    cv2.imencode(".png", r["mask"].astype(np.uint8) * 255)[1].tofile(str(path))
    print(f"{path.name}: {name} — 패턴 면적 {r['pattern']['area_fraction'] * 100:.1f}%")

00_mask.png: 흰 배경 문서 — 패턴 면적 0.8%
01_mask.png: 작은 줄무늬 조각 + 글자형 블록 — 패턴 면적 3.5%
02_mask.png: 좌흑 / 우백 — 패턴 면적 0.0%
03_mask.png: 포화 적색 45% (정지) — 패턴 면적 0.0%
04_mask.png: 세로 줄무늬 전체 — 패턴 면적 100.0%
05_mask.png: 동심원 전체 — 패턴 면적 99.9%
06_mask.png: 화면 왼쪽 절반 줄무늬 — 패턴 면적 50.0%


## 한계와 다음 단계

1. 주사선은 가로·세로 두 방향이다. 줄무늬가 정확히 45°로 매우 가늘면(1px 미만 간격) 표본화 한계로 놓칠 수 있다.
2. 휘도는 sRGB·흰색 200 cd/m² 가정이다. 실제 표시 장치 밝기·HDR은 반영하지 않는다.
3. 패턴 면적 기준 25%/40%. 적용 표준·정책 버전을 결정해 `docs/EXPERIMENT_PLAN.md`에 기록해야 한다.
4. 다음 단계: 이 특징값을 프레임마다 계산해 **프레임 간 휘도·적색 변화(깜빡임)** 판정기로 확장한다.
5. 저장소 버전과의 일치 여부는 `tests/test_static_notebook.py`가 검사한다(이 노트북 실행에는 필요 없음).